# 01 — pkl 구조와 메타 재집계

**무엇을 확인하는가**

1. 셀 pkl 하나의 스키마를 눈으로 본다
2. **전 서브셋의 메타 필드를 다시 센다**
3. 결과를 `findings/recount.json` 에 정규화 JSON 으로 남긴다

3번이 `META-` 레코드의 `code` 슬롯 근거가 됩니다.

## 순서 — 데이터 먼저, 논문 나중

```
1. 배포 데이터 재집계     ← 이 노트북
2. 논문 숫자와 대조       Table 1 · Abstract · 부록 A
3. 갈리는 것만 레코드화   META- 목록은 여기서 확정된다
4. 논문 근거 찾기         부록까지. 못 찾으면 `조사했으나불명`
```

순서를 뒤집으면 항목이 폭발하고, 무엇을 찾아야 하는지 모른 채 논문을 읽게
됩니다. **59 나 421 을 맞추려고 세는 방식을 고르지 마십시오.** 세고 나서
갈리면 갈린다고 적습니다.

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

import pickle
from collections import Counter

from verify import load_config, write_json, FINDINGS

config = load_config()
EXTRACT = Path(config["EXTRACT_DIR"])
print("데이터 루트:", EXTRACT, "[있음]" if EXTRACT.exists() else "[없음]")

## 1. 서브셋 목록

폴더 이름이 배포 zip 이름을 따릅니다 (`NA-ion`, `ZN-coin`). 상위 스크립트가
쓰는 `dataset_name`(`NAion`, `ZNcoin`)과 표기가 다릅니다.

In [ ]:
subset_dirs = sorted(
    p for p in EXTRACT.iterdir()
    if p.is_dir() and p.name not in ("Life labels", "READMEs")
)
for path in subset_dirs:
    n = sum(1 for f in path.iterdir() if f.suffix == ".pkl")
    print(f"  {path.name:14} {n:4} cells")
print(f"\n서브셋 {len(subset_dirs)}개")

## 2. 셀 하나의 스키마

무엇이 들어 있는지 직접 보십시오. 아래 재집계는 이 필드 이름에 기대고
있습니다.

In [ ]:
sample_dir = subset_dirs[0]
sample_file = sorted(f for f in sample_dir.iterdir() if f.suffix == ".pkl")[0]
with open(sample_file, "rb") as f:
    sample = pickle.load(f)

print(sample_file.name)
print()
for key, value in sample.items():
    if key == "cycle_data":
        print(f"  {key:28} list[{len(value)}]")
        for ckey, cvalue in value[0].items():
            kind = f"list[{len(cvalue)}]" if isinstance(cvalue, (list, tuple)) else repr(cvalue)
            print(f"      {ckey:24} {kind}")
    else:
        print(f"  {key:28} {value!r}")

## 3. 전 서브셋 재집계

세는 것:

- `form_factor` 고유값 → `META-002` (포맷 8종)
- `(cathode, anode, electrolyte)` 조합 → `META-001` (화학계 59종)
- `nominal_capacity_in_Ah` 분포
- `SOC_interval` 분포 → `LAB-005`
- `charge_protocol` · `discharge_protocol` 분포와 **`multi` 비율** → `META-004`
- `temperature_in_C` **보유 셀 비율** → `META-003`
- 서브셋별 셀 수 · 사이클 수 → `META-005`

`multi` 비율이 중요합니다. 다단 프로토콜이면 값이 `"multi"` 라 **데이터만으로
구별되지 않습니다.** 그 비율이 높으면 421 이라는 숫자를 데이터에서 재현할 수
없다는 뜻이고, 그것이 결론입니다.

In [ ]:
def field(data, *names):
    for name in names:
        if name in data:
            return data[name]
    return None


recount = {
    "status": "집계완료",
    "dataset_root": str(EXTRACT),
    "by_subset": {},
    "form_factor": Counter(),
    "chemistry_triplet": Counter(),
    "charge_protocol": Counter(),
    "discharge_protocol": Counter(),
    "soc_interval": Counter(),
    "nominal_capacity_in_Ah": Counter(),
    "missing_fields": Counter(),
}
temp_with, temp_total = 0, 0
total_cells, total_cycles = 0, 0

for path in subset_dirs:
    files = sorted(f for f in path.iterdir() if f.suffix == ".pkl")
    cycles_here = 0
    for file in files:
        with open(file, "rb") as f:
            data = pickle.load(f)

        cycles = len(data.get("cycle_data", []))
        cycles_here += cycles

        recount["form_factor"][str(field(data, "form_factor"))] += 1
        triplet = (
            str(field(data, "cathode_material")),
            str(field(data, "anode_material")),
            str(field(data, "electrolyte_material")),
        )
        recount["chemistry_triplet"][" | ".join(triplet)] += 1
        recount["charge_protocol"][str(field(data, "charge_protocol"))] += 1
        recount["discharge_protocol"][str(field(data, "discharge_protocol"))] += 1
        recount["soc_interval"][str(field(data, "SOC_interval"))] += 1
        recount["nominal_capacity_in_Ah"][str(field(data, "nominal_capacity_in_Ah"))] += 1

        temp_total += 1
        if field(data, "temperature_in_C") is not None:
            temp_with += 1

        for name in ("form_factor", "cathode_material", "anode_material",
                     "electrolyte_material", "charge_protocol", "SOC_interval",
                     "nominal_capacity_in_Ah", "temperature_in_C"):
            if name not in data:
                recount["missing_fields"][name] += 1

    recount["by_subset"][path.name] = {"cells": len(files), "cycles": cycles_here}
    total_cells += len(files)
    total_cycles += cycles_here
    print(f"  {path.name:14} {len(files):4} cells  {cycles_here:8} cycles")

print(f"\n총 {total_cells} cells / {total_cycles} cycles")

## 4. 논문 숫자와 나란히

여기서 나온 숫자를 `papers/NOTES.md` 의 표에 옮겨 적고 논문 Table 1 과
대조하십시오. **갈리는 것만** `findings/registry.yaml` 에 레코드로
추가합니다.

In [ ]:
def multi_ratio(counter):
    total = sum(counter.values())
    return round(counter.get("multi", 0) / total, 4) if total else None


print("포맷 (form_factor) 고유값:", len(recount["form_factor"]))
for value, count in recount["form_factor"].most_common():
    print(f"    {value:24} {count}")

print("\n화학계 조합 고유값:", len(recount["chemistry_triplet"]))
for value, count in recount["chemistry_triplet"].most_common(15):
    print(f"    {value[:60]:60} {count}")
if len(recount["chemistry_triplet"]) > 15:
    print(f"    ... 외 {len(recount['chemistry_triplet']) - 15}개")

print("\n충전 프로토콜 고유값:", len(recount["charge_protocol"]),
      "  multi 비율:", multi_ratio(recount["charge_protocol"]))
print("방전 프로토콜 고유값:", len(recount["discharge_protocol"]),
      "  multi 비율:", multi_ratio(recount["discharge_protocol"]))
print("SOC_interval 고유값:", len(recount["soc_interval"]))
print(f"temperature_in_C 보유: {temp_with}/{temp_total} "
      f"({round(temp_with / temp_total, 4) if temp_total else None})")

print("\n논문 대조 (papers/NOTES.md 에 옮겨 적으십시오):")
print("    화학계   논문 59   재집계", len(recount["chemistry_triplet"]))
print("    포맷     논문  8   재집계", len(recount["form_factor"]))
print("    프로토콜 논문 421  재집계", len(recount["charge_protocol"]))
print("    셀       논문 990  재집계", total_cells)
print()
print("갈리면 갈린다고 적으십시오. 세는 방식을 바꿔 맞추지 마십시오.")

## 5. `findings/recount.json` 저장

`normalized=True` 로 씁니다 — LOCK 의 해시 대상과 같은 바이트가 되도록.

In [ ]:
from datetime import datetime

recount_out = {
    "status": "집계완료",
    "generated": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "dataset_root": str(EXTRACT),
    "totals": {
        "subsets": len(subset_dirs),
        "cells": total_cells,
        "cycles": total_cycles,
    },
    "by_subset": recount["by_subset"],
    "form_factor": dict(recount["form_factor"]),
    "chemistry_triplet": dict(recount["chemistry_triplet"]),
    "nominal_capacity_in_Ah": {
        "unique_count": len(recount["nominal_capacity_in_Ah"]),
        "histogram": dict(recount["nominal_capacity_in_Ah"]),
    },
    "soc_interval": dict(recount["soc_interval"]),
    "charge_protocol": {
        "unique_count": len(recount["charge_protocol"]),
        "multi_ratio": multi_ratio(recount["charge_protocol"]),
        "values": dict(recount["charge_protocol"]),
    },
    "discharge_protocol": {
        "unique_count": len(recount["discharge_protocol"]),
        "multi_ratio": multi_ratio(recount["discharge_protocol"]),
        "values": dict(recount["discharge_protocol"]),
    },
    "temperature_in_C": {
        "cells_with_field": temp_with,
        "cells_total": temp_total,
        "ratio": round(temp_with / temp_total, 4) if temp_total else None,
        "values": {},
    },
    "missing_fields": dict(recount["missing_fields"]),
    "notes": [],
}

write_json(FINDINGS / "recount.json", recount_out, normalized=True)
print("저장:", FINDINGS / "recount.json")
print("\n이제 python run.py lock-init 이 nb01 항목을 채울 수 있습니다.")